# 03 — Análisis Descriptivo
**Proyecto:** OpenFarma — Sistema de Alerta Temprana de Desabastecimiento Farmacéutico  

Este notebook presenta estadísticas descriptivas y análisis de correlaciones entre las variables del mercado farmacéutico colombiano y el historial de alertas INVIMA.

**Pregunta central:** ¿Qué características del mercado (CUM) están correlacionadas con el historial de desabastecimiento (INVIMA)?

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DB_PATH = Path('../backend/openfarma.db')
conn = sqlite3.connect(DB_PATH)

## 1. Construcción del dataset analítico

Combinamos métricas de estructura de mercado (del CUM) con el historial de alertas INVIMA para cada código ATC.

In [ ]:
# Features de estructura de mercado por ATC5
mercado = pd.read_sql("""
    SELECT 
        SUBSTR(atc_normalizado, 1, 5) as atc5,
        COUNT(DISTINCT expediente_cum || consecutivo_cum) as n_presentaciones,
        COUNT(DISTINCT titular_registro) as n_titulares,
        SUM(CASE WHEN estado_cum = 'Activo' THEN 1 ELSE 0 END) as activos,
        SUM(CASE WHEN estado_cum != 'Activo' THEN 1 ELSE 0 END) as inactivos,
        SUM(CASE WHEN tipo_formula = 'combinado' THEN 1 ELSE 0 END) as combinados
    FROM cum_normalizado
    WHERE atc_normalizado IS NOT NULL AND LENGTH(atc_normalizado) >= 5
    GROUP BY atc5
    HAVING activos > 0
""", conn)

mercado['tasa_inactivacion'] = mercado['inactivos'] / (mercado['activos'] + mercado['inactivos'])
mercado['monopolio'] = (mercado['n_titulares'] == 1).astype(int)

print(f'ATCs analizados: {len(mercado):,}')
print(f'\nEstadísticas de concentración de mercado:')
print(mercado[['n_titulares', 'n_presentaciones', 'tasa_inactivacion']].describe().round(2))

In [ ]:
# Features INVIMA por ATC5
SEVERIDAD = {
    'DESABASTECIDO': 5, 'EN_RIESGO': 4, 'MONITORIZACION': 3,
    'NO_COMERCIALIZADO': 2, 'DESCONTINUADO': 1
}

invima_raw = pd.read_sql("""
    SELECT SUBSTR(atc, 1, 5) as atc5, estado, mes, anio
    FROM invima_seguimiento
    WHERE atc IS NOT NULL AND LENGTH(atc) >= 5
""", conn)

invima_raw['severidad'] = invima_raw['estado'].map(SEVERIDAD).fillna(0)

invima_agg = invima_raw.groupby('atc5').agg(
    meses_monitoreado=('severidad', 'count'),
    sev_max=('severidad', 'max'),
    sev_mean=('severidad', 'mean'),
    n_desabastecido=('estado', lambda x: (x == 'DESABASTECIDO').sum()),
    n_monitorizacion=('estado', lambda x: (x == 'MONITORIZACION').sum()),
).reset_index()

print(f'ATCs con historial INVIMA: {len(invima_agg):,}')
print(f'\nDistribución de severidad máxima:')
print(invima_agg['sev_max'].value_counts().sort_index().to_string())

## 2. Correlaciones entre estructura de mercado y alertas INVIMA

In [ ]:
# Unir datasets
df = mercado.merge(invima_agg, on='atc5', how='inner')
print(f'Dataset analítico: {len(df):,} ATCs con datos de ambas fuentes')

# Matriz de correlación
features_corr = [
    'n_titulares', 'n_presentaciones', 'tasa_inactivacion', 'monopolio',
    'meses_monitoreado', 'sev_max', 'sev_mean', 'n_desabastecido'
]

corr_matrix = df[features_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    ax=ax, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Matriz de correlación: estructura de mercado × historial INVIMA', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('../reports/figures/correlaciones.png', bbox_inches='tight')
plt.show()
print('Figura guardada en reports/figures/correlaciones.png')

## 3. Análisis de monopolio vs. desabastecimiento

In [ ]:
# ¿Los medicamentos con un solo titular se desabastecen más?
df['tiene_alerta'] = (df['sev_max'] >= 3).astype(int)

comp = df.groupby('monopolio').agg(
    total=('atc5', 'count'),
    con_alerta=('tiene_alerta', 'sum'),
    pct_alerta=('tiene_alerta', 'mean'),
    sev_max_promedio=('sev_max', 'mean'),
).reset_index()
comp['monopolio'] = comp['monopolio'].map({0: 'Competencia (≥2 titulares)', 1: 'Monopolio (1 titular)'})

print('Monopolio vs. competencia — riesgo de desabastecimiento:')
print(comp.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# % con alerta
axes[0].bar(comp['monopolio'], comp['pct_alerta'] * 100, color=['#10b981', '#ef4444'], edgecolor='white')
axes[0].set_title('% de ATCs con alerta INVIMA alguna vez', fontsize=11)
axes[0].set_ylabel('%')
for i, (_, row) in enumerate(comp.iterrows()):
    axes[0].text(i, row['pct_alerta'] * 100 + 0.5, f"{row['pct_alerta']*100:.1f}%", ha='center', fontsize=11)

# Severidad promedio
axes[1].bar(comp['monopolio'], comp['sev_max_promedio'], color=['#10b981', '#ef4444'], edgecolor='white')
axes[1].set_title('Severidad máxima promedio (0-5)', fontsize=11)
axes[1].set_ylabel('Severidad')
axes[1].set_ylim(0, 5)

plt.tight_layout()
plt.show()

## 4. Persistencia de alertas — ¿Las alertas se repiten?

In [ ]:
# Análisis de persistencia: ¿cuántos meses consecutivos dura una alerta?
invima_timeline = pd.read_sql("""
    SELECT principio_activo, anio*100+mes as periodo, estado
    FROM invima_seguimiento
    WHERE estado IN ('MONITORIZACION', 'EN_RIESGO', 'DESABASTECIDO')
    ORDER BY principio_activo, periodo
""", conn)

# Calcular duración de episodios por principio activo
duraciones = (
    invima_timeline
    .groupby('principio_activo')
    .size()
    .reset_index(name='meses_en_alerta')
)

print(f'Principios activos que han estado en alerta: {len(duraciones):,}')
print(f'\nDistribución de meses en alerta por principio activo:')
print(duraciones['meses_en_alerta'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
duraciones['meses_en_alerta'].clip(upper=17).hist(bins=17, ax=ax, color='#f59e0b', edgecolor='white', range=(0.5, 17.5))
ax.set_title('Duración de alertas INVIMA por principio activo\n(meses acumulados en monitorización/riesgo/desabastecido)', fontsize=11)
ax.set_xlabel('Meses en alerta (de 17 posibles)')
ax.set_ylabel('Número de principios activos')
plt.tight_layout()
plt.show()

recurrentes = (duraciones['meses_en_alerta'] >= 6).sum()
print(f'\nPrincipios activos en alerta ≥6 meses ("crónicos"): {recurrentes} ({recurrentes/len(duraciones)*100:.1f}%)')

conn.close()
print('\nAnálisis descriptivo completado. Figuras en reports/figures/')